# Adult Minimization Privacy Controls Test

This notebook compares:
- minimization with a utility-only configuration
- minimization with all privacy features (DP + homogeneity guard + privacy floor + risk-driven enforcement)

The notebook uses only numeric Adult features to keep the run simple and aligned with the existing minimization examples.

In [14]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

from apt.minimization.minimizer import GeneralizeToRepresentative
from apt.utils.dataset_utils import get_adult_dataset_pd
from apt.utils.datasets import ArrayDataset

np.random.seed(7)

In [15]:
# Load Adult data and keep numeric features only
(x_train_full, y_train_full), (x_test_full, y_test_full) = get_adult_dataset_pd()

numeric_features = [
    "age",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]

x_train_num = x_train_full[numeric_features].copy()
x_test_num = x_test_full[numeric_features].copy()

# Keep the notebook reasonably fast while preserving class balance approximately.
train_size = min(4000, len(x_train_num))
test_size = min(2000, len(x_test_num))

train_idx = x_train_num.sample(n=train_size, random_state=7).index
test_idx = x_test_num.sample(n=test_size, random_state=7).index

x_train = x_train_num.loc[train_idx].reset_index(drop=True)
y_train = y_train_full.loc[train_idx].reset_index(drop=True)
x_test = x_test_num.loc[test_idx].reset_index(drop=True)
y_test = y_test_full.loc[test_idx].reset_index(drop=True)

feature_names = list(x_train.columns)
print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)
print("Features:", feature_names)

Train shape: (4000, 5)
Test shape: (2000, 5)
Features: ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']


In [16]:
base_model = DecisionTreeClassifier(random_state=0, max_depth=8, min_samples_split=10, min_samples_leaf=5)
base_model.fit(x_train, y_train)

base_train_accuracy = base_model.score(x_train, y_train)
base_test_accuracy = base_model.score(x_test, y_test)
print("Base model train accuracy:", base_train_accuracy)
print("Base model test accuracy:", base_test_accuracy)

y_train_pred = base_model.predict(x_train)
y_test_pred = base_model.predict(x_test)

Base model train accuracy: 0.843
Base model test accuracy: 0.822


In [17]:
train_dataset = ArrayDataset(x_train, y_train_pred, features_names=feature_names)
test_dataset = ArrayDataset(x_test, features_names=feature_names)

In [ ]:
# Fit minimizer with a utility-only configuration (on encoded Adult data)
minimizer_base = GeneralizeToRepresentative(
    estimator=base_model,
    target_accuracy=0.95,
)
minimizer_base.fit(dataset=train_dataset)

# test_dataset.get_samples() is numpy, wrap with encoded feature names
X_test_enc_df = pd.DataFrame(test_dataset.get_samples(), columns=feature_names)

# minimizer returns generalized samples
x_test_base = minimizer_base.transform(X_test_enc_df)

# predict using the same encoded feature names
x_test_base_df = pd.DataFrame(x_test_base, columns=feature_names)
pred_base = base_model.predict(x_test_base_df)

agreement_base = np.mean(pred_base == y_test_pred)

print("Utility-only level:", minimizer_base._level)
print("Utility-only removed_count:", minimizer_base._removed_count)
print("Utility-only agreement:", agreement_base)
print("Utility-only NCP:", minimizer_base.ncp.generalizations_score)


Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.985000

Improving generalizations
Pruned tree to level: 1, new relative accuracy: 0.985000
Pruned tree to level: 2, new relative accuracy: 0.985000
Pruned tree to level: 3, new relative accuracy: 0.985000
Pruned tree to level: 4, new relative accuracy: 0.985000
Pruned tree to level: 5, new relative accuracy: 0.984375
Pruned tree to level: 6, new relative accuracy: 0.981875
Pruned tree to level: 7, new relative accuracy: 0.976875
Pruned tree to level: 8, new relative accuracy: 0.974375
Pruned tree to level: 9, new relative accuracy: 0.955000
Pruned tree to level: 10, new relative accuracy: 0.951250
Accuracy falling below target accuracy, restoring previous state
Utility-only level: 10
Utility-only removed_count: 0
Utility-only agreement: 0.955
Utility-only NCP: 5.12857142857143


In [31]:
# Fit minimizer with privacy controls enabled
minimizer_priv = GeneralizeToRepresentative(
    estimator=base_model,
    target_accuracy=0.95,

    # DP on surrogate thresholds
    epsilon=1.0,
    delta=0.0,
    dp_random_state=7,

    # Relative privacy floor
    privacy_floor_alpha=0.60,
    privacy_enforcement="auto",

    # Homogeneity guard
    homogeneity_min_cell_size=5,
    homogeneity_min_label_diversity=2,
    homogeneity_enforcement="auto",
    max_homogeneity_prune_level=7,
    homogeneity_accuracy_tolerance=0.05,

    # Risk-driven enforcement
    risk_attack_type="membership_classification",
    risk_max_risk=0.10,
    risk_max_member_auc=0.9,
    risk_max_non_member_auc=0.9,
    risk_require_no_warning=True,
    risk_enforcement="auto",
    max_risk_prune_level=10,
    risk_accuracy_tolerance=0.10,
)
minimizer_priv.fit(dataset=train_dataset)



x_test_priv_df = pd.DataFrame(test_dataset.get_samples(), columns=feature_names)
x_test_priv = minimizer_priv.transform(x_test_priv_df)
pred_priv = base_model.predict(x_test_priv)
agreement_priv = np.mean(pred_priv == y_test_pred)

print("Privacy-controlled level:", minimizer_priv._level)
print("Privacy-controlled removed_count:", minimizer_priv._removed_count)
print("Privacy-controlled agreement:", agreement_priv)
print("Privacy-controlled NCP:", minimizer_priv.ncp.generalizations_score)


Differential Privacy is enabled
Epsilon:  1.0
Delta:  0.0
[DP] node=0 feature=education-num orig=12.500000 noisy=11.849378
[DP] node=1 feature=capital-gain orig=5095.500000 noisy=758.064027
[DP] node=2 feature=capital-loss orig=1805.000000 noisy=1635.838305
[DP] node=3 feature=hours-per-week orig=64.000000 noisy=59.749270
[DP] node=5 feature=age orig=35.500000 noisy=32.333640
[DP] node=7 feature=age orig=46.500000 noisy=43.333640
[DP] node=8 feature=education-num orig=9.500000 noisy=8.849378
[DP] node=12 feature=capital-loss orig=1978.500000 noisy=1809.338305
[DP] node=13 feature=age orig=38.500000 noisy=35.333640
[DP] node=14 feature=hours-per-week orig=49.000000 noisy=44.749270
[DP] node=15 feature=capital-loss orig=1975.500000 noisy=1806.338305
[DP] node=18 feature=age orig=36.000000 noisy=32.833640
[DP] node=22 feature=hours-per-week orig=74.000000 noisy=69.749270
[DP] node=23 feature=hours-per-week orig=49.000000 noisy=44.749270
[DP] node=25 feature=capital-loss orig=2176.500000 n

In [34]:
# Compare outputs
comparison = pd.DataFrame({
    "setting": ["utility_only", "privacy_controlled"],
    "agreement_with_base_model": [agreement_base, agreement_priv],
    "ncp_generalizations_score": [
        minimizer_base.ncp.generalizations_score,
        minimizer_priv.ncp.generalizations_score,
    ],
})
comparison

,setting,agreement_with_base_model,ncp_generalizations_score
0,utility_only,0.9550,5.128571
1,privacy_controlled,0.9395,7.205246
